# AN8 — Validated Metrics and Curated Reporting Tables

**Status: PREPARATION ONLY — AN8 has not started.** This notebook is a staged execution guide. It contains no retained execution evidence and must not be run with **Run All**.

The local repository has not yet been synchronized and these files have not yet been confirmed in the user's local working tree. Execute only after the branch has been reviewed and synchronized.

## Analytical architecture

1. Run the approved read-only SQL extracts once, with explicit gates.
2. Retain each complete result set as a local CSV plus a non-sensitive manifest under `outputs/an8/extracts/`.
3. Review and reconcile the retained extracts.
4. Only after explicit approval, load the validated extracts into curated PostgreSQL reporting tables.
5. Build all charts, tables, and Power BI sources from those curated tables or retained exports — **never by rerunning the large source scans for each visualization**.

The future curated tables are a separate persistent-database change and are not created by this notebook or the read-only runner. No metric, KPI, threshold, or composite score is approved.

## 0. Environment and helper imports

This cell only imports the project runner and standard libraries. It does not connect to PostgreSQL.

In [2]:
from pathlib import Path
import sys
import pandas as pd

# Locate the repository root from the current Jupyter directory.
PROJECT_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "scripts" / "run_an8.py").is_file()
        and (parent / "src" / "analytics" / "db.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the OULAD repository root. "
        "Start Jupyter from the repository or one of its subdirectories."
    )

# Make the repository root available to Python imports.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_an8 import preflight, run_one, export_results

AN8_EXPORT_DIR = PROJECT_ROOT / "outputs" / "an8" / "extracts"
query_results = {}

print("Project root:", PROJECT_ROOT)
print("AN8 runner import: PASS")

Project root: G:\OULAD-EdTech-Learning-Analytics-starter\OULAD-EdTech-Learning-Analytics
AN8 runner import: PASS


## 1. Database identity and read-only preflight

Run this cell first. It must confirm database `oulad_analytics`, `transaction_read_only=on`, and `default_transaction_read_only=on`. A failed preflight blocks every AN8 query.

In [3]:
preflight_result = preflight()


AN8 — read-only database preflight
  database_name database_user current_schema server_version transaction_read_only default_transaction_read_only statement_timeout lock_timeout
oulad_analytics      postgres         public           18.6                    on                            on              2min           5s
database_name: PASS
transaction_read_only: PASS
default_transaction_read_only: PASS


## 2. AN8.01 — Lightweight source relation inventory

This query reads PostgreSQL catalogs only. Estimated row counts are planner statistics, not exact counts. It deliberately avoids a full `COUNT(*)` scan of `staging.student_vle`.

In [4]:
an801 = run_one("AN8.01")
query_results["AN8.01"] = an801
an801["data"]


AN8.01 — source readiness gate
all_expected_relations_returned: PASS
all_expected_relations_exist: PASS
AN8.01: 0.205 s; 1 result set(s); 7 displayed rows; no truncation


,source_relation,relation_exists,estimated_rows,last_analyze,last_autoanalyze
0,staging.assessments,True,206,None,None
1,staging.courses,True,0,None,None
2,staging.student_assessment,True,173912,None,None
3,staging.student_info,True,32593,None,None
4,staging.student_registration,True,32593,None,None
5,staging.student_vle,True,10654140,None,None
6,staging.vle,True,6364,None,None


## 3. AN8.02 — Student-presentation grain and key-set gate

This is a required stop/go gate. The runner checks both sources for duplicate composite keys, NULL key components, row-to-distinct-key agreement, and bidirectional key-set differences. If any check fails, AN8.03 and all activity extraction are blocked.

Review the complete result before continuing.

In [5]:
an802 = run_one("AN8.02")
query_results["AN8.02"] = an802
an802["data"]


AN8.02 — student-presentation grain gate
    check_type            check_name  source_rows  distinct_keys  duplicate_key_groups  null_key_rows  key_count
KEY_DIFFERENCE INFO_NOT_REGISTRATION          NaN            NaN                   NaN            NaN        0.0
KEY_DIFFERENCE REGISTRATION_NOT_INFO          NaN            NaN                   NaN            NaN        0.0
  SOURCE_GRAIN          student_info      32593.0        32593.0                   0.0            0.0        NaN
  SOURCE_GRAIN  student_registration      32593.0        32593.0                   0.0            0.0        NaN
no_duplicate_student_info_keys: PASS
no_duplicate_registration_keys: PASS
student_info_rows_equal_distinct_keys: PASS
registration_rows_equal_distinct_keys: PASS
no_null_key_rows: PASS
no_key_set_differences: PASS
AN8.02: 0.635 s; 1 result set(s); 4 displayed rows; no truncation


,check_type,check_name,source_rows,distinct_keys,duplicate_key_groups,null_key_rows,key_count
0,KEY_DIFFERENCE,INFO_NOT_REGISTRATION,NaN,NaN,NaN,NaN,0.0
1,KEY_DIFFERENCE,REGISTRATION_NOT_INFO,NaN,NaN,NaN,NaN,0.0
2,SOURCE_GRAIN,student_info,32593.0,32593.0,0.0,0.0,NaN
3,SOURCE_GRAIN,student_registration,32593.0,32593.0,0.0,0.0,NaN


## 4. AN8.03 — Outcome denominator profile

This query is allowed only after AN8.02 passes. It describes retrospective `final_result` categories and registration matching; it does not define a risk score or predictive feature.

In [6]:
an803 = run_one("AN8.03")
query_results["AN8.03"] = an803
an803["data"]

\nAN8.03 — outcome denominator reconciliation
outcome_denominator_matches_student_info: PASS
registration_matches_equal_validated_cohort: PASS
AN8.03: 0.07 s; 1 result set(s); 4 displayed rows; no truncation


,final_result,student_presentation_rows,registration_matched_rows,distinct_student_ids_in_category
0,Distinction,3024,3024,2903
1,Fail,7052,7052,6810
2,Pass,12361,12361,11881
3,Withdrawn,10156,10156,9155


## 5. AN8.04 — Course and assessment context

AN8.04 returns **two result sets**. The runner checks truncation independently for each result set. Due dates and weights do not establish score or feedback availability.

In [ ]:
an804 = run_one("AN8.04")
query_results["AN8.04"] = an804
for item in an804["result_sets"]:
    display(item["data"])

## 6. AN8.10–AN8.12 — Lightweight descriptive context

Run these only after the grain gate passes. Each query has its own result set and must be reviewed before the activity extract. These are descriptive context, not approved KPIs.

In [ ]:
for query_id in ("AN8.10", "AN8.11", "AN8.12"):
    query_results[query_id] = run_one(query_id)
    print(query_id)
    display(query_results[query_id]["data"])

## 7. AN8.20 — Consolidated activity extract (single high-cost scan)

**Run this query once, alone.** It consolidates the activity measures that were previously spread across AN8.20–AN8.23 into one pass over `staging.student_vle`.

The output contains one row per registered student-presentation, including:
- R0 source activity-row and click totals;
- pre-course, in-course, and post-course row/click partitions;
- in-course active days and resource breadth;
- post-registration active days, resource breadth, exposure days, and active-day rate.

The numerator of the post-registration active-day rate is restricted to dates from `max(date_registration, 0)` through course end. Missing registration dates remain NULL for post-registration measures. The extract does not truncate activity at unregistration.

The runner checks the output row count and composite-key uniqueness against the validated AN8.02 cohort. Stop if the gate fails.

In [ ]:
activity_extract = run_one("AN8.20")
query_results["AN8.20"] = activity_extract
activity_frame = activity_extract["data"]
print(activity_frame.shape)
display(activity_frame.head())

## 8. Persist the reviewed extract locally

After inspecting the complete AN8.20 output and its gate, export the retained DataFrames to CSV with a JSON manifest. This is local file output only; it does not create PostgreSQL objects.

Do not export if a result set is truncated or the cohort gate failed.

In [ ]:
manifest_path = export_results(query_results, AN8_EXPORT_DIR)
manifest_path

## 9. Python-only reconciliation from retained extracts

All calculations in this section use in-memory results or exported CSVs. No SQL or database connection is issued. Reconcile the course-relative row totals and registered population before creating any visualization.

In [ ]:
activity = pd.read_csv(AN8_EXPORT_DIR / "AN8_20_result_01.csv")registered_rows = len(activity)registration_denominator = int(query_results['AN8.02']['result_sets'][0]['data'].loc[lambda frame: frame['check_type'].eq('SOURCE_GRAIN') & frame['check_name'].eq('student_registration'), 'source_rows'].iloc[0])source_total = int(activity["total_source_activity_rows"].iloc[0])matched_source_rows = int(activity["matched_registered_activity_rows"].iloc[0])unmatched_source_rows = int(activity["unmatched_activity_rows"].iloc[0])source_clicks = float(activity["total_source_recorded_clicks"].iloc[0])matched_clicks = float(activity["matched_registered_recorded_clicks"].iloc[0])unmatched_clicks = float(activity["unmatched_recorded_clicks"].iloc[0])matched_partition_total = int(activity[[    "pre_course_activity_rows",    "in_course_activity_rows",    "post_course_activity_rows",    "no_course_window_activity_rows"]].sum().sum())source_partition_total = int(activity[[    "all_source_pre_course_activity_rows",    "all_source_in_course_activity_rows",    "all_source_post_course_activity_rows",    "all_source_no_course_window_activity_rows"]].iloc[0].sum())matched_click_partition_total = float(activity[[    "pre_course_recorded_clicks",    "in_course_recorded_clicks",    "post_course_recorded_clicks",    "no_course_window_recorded_clicks"]].sum().sum())source_click_partition_total = float(activity[[    "all_source_pre_course_recorded_clicks",    "all_source_in_course_recorded_clicks",    "all_source_post_course_recorded_clicks",    "all_source_no_course_window_recorded_clicks"]].iloc[0].sum())exposed = activity["post_registration_exposure_days"].notna()checks = {    "row_count_matches_an802_registration_denominator": registered_rows == registration_denominator,    "non_null_student_presentation_key": not activity[["code_module", "code_presentation", "id_student"]].isna().any().any(),    "one_row_per_registered_student_presentation": registered_rows == len(        activity[["code_module", "code_presentation", "id_student"]].drop_duplicates()    ),    "registered_activity_rows_reconcile": int(activity["all_observed_activity_rows"].sum()) == matched_source_rows,    "registered_temporal_partition_reconciles": matched_partition_total == matched_source_rows,    "all_source_temporal_partition_reconciles": source_partition_total == source_total,    "source_rows_equal_matched_plus_unmatched": source_total == matched_source_rows + unmatched_source_rows,    "clicks_equal_matched_plus_unmatched": abs(source_clicks - matched_clicks - unmatched_clicks) < 1e-9,    "registered_click_partition_reconciles": abs(matched_click_partition_total - matched_clicks) < 1e-9,    "source_click_partition_reconciles": abs(source_click_partition_total - source_clicks) < 1e-9,    "active_days_are_nonnegative": bool((activity["in_course_active_days"] >= 0).all()),    "active_days_do_not_exceed_exposure": bool(        (activity.loc[exposed, "post_registration_active_days"] >= 0).all()        and (activity.loc[exposed, "post_registration_active_days"]             <= activity.loc[exposed, "post_registration_exposure_days"]).all()    ),}for name, passed in checks.items():    print(f"{name}: {'PASS' if passed else 'FAIL'}")if not all(checks.values()):    raise RuntimeError("AN8.20 reconciliation failed; stop before visualizations or mart design.")pd.DataFrame([{    "registered_student_presentations": registered_rows,    "source_activity_rows": source_total,    "source_recorded_clicks": source_clicks,    "matched_registered_recorded_clicks": matched_clicks,    "unmatched_recorded_clicks": unmatched_clicks,    "matched_registered_activity_rows": matched_source_rows,    "unmatched_activity_rows": unmatched_source_rows,    "registered_temporal_partition_rows": matched_partition_total,    "all_source_temporal_partition_rows": source_partition_total,}])

## 10. Visualization source contract

Every chart must have its own preceding Markdown explanation. Visualization cells read only the curated reporting tables or retained files under `outputs/an8/`. They must not call `run_one`, `run_batch`, `run_query`, or connect to PostgreSQL.

Illustrative starter below is intentionally Python-only and uses the retained extract.

In [ ]:
import matplotlib.pyplot as plt

plot_data = activity.groupby("code_module", as_index=False)["in_course_active_days"].median()
ax = plot_data.plot(kind="bar", x="code_module", y="in_course_active_days", legend=False)
ax.set_title("Median in-course active days by module")
ax.set_xlabel("Module")
ax.set_ylabel("Median student-presentation active days")
plt.tight_layout()

## 11. Curated reporting tables — approval gate

The intended downstream design is to load the validated retained extracts into a small set of persistent, documented reporting tables (for example, a student-presentation activity fact and course/assessment dimensions), then build all charts and Power BI datasets from those tables.

**No table, view, materialized view, index, schema, or DDL is created here.** Before deployment, the project owner must review and explicitly approve:
- table names, schemas, columns, keys, and data types;
- CSV-to-table load method and reconciliation checks;
- refresh/replacement policy and provenance manifest;
- retention, access, and privacy controls;
- exact row/key/count/denominator acceptance criteria.

After approval, prepare a separate DDL/load work order. It must load the already validated extracts, not rescan `staging.student_vle` to recreate chart inputs.

## 12. Deferred and retired query IDs

- AN8.21–AN8.23 are retired as separate source scans; their required measures are consolidated into AN8.20.
- AN8.30–AN8.32 remain deferred pending a validated reporting-table contract.
- AN8.40–AN8.41 are planning/validation SELECTs only; they do not deploy database objects.

## 13. Final status

Until local synchronization, controlled SQL execution, output review, reconciliation, and approval are complete, AN8 remains **NOT STARTED / NOT VALIDATED**. AN9 Power BI cannot begin from unapproved or unpersisted candidate metrics.